# Geocoding — Place Names to Coordinates

Most real workflows start with a city name or address, not a lat/lon pair.
`idd_travel_time.locations` resolves place names via OpenStreetMap Nominatim
(free, no API key, global coverage).

**What's covered**
1. `geocode()` — single place to (lat, lon)
2. `country_hint` — bias results to a specific country
3. `TravelTimeRequest.from_place()` — geocode then build request in one call
4. `geocode_many()` — batch geocoding with graceful failure handling
5. The geocoding cache — in-memory and on-disk
6. Rate limiting and usage policy

> Nominatim's usage policy requires ≤ 1 request/second. The library
> enforces this automatically — no action needed.


## 1 · Single place lookup

In [ ]:
from idd_travel_time.locations import geocode

lat, lon = geocode("Kenyatta National Hospital, Nairobi")
print(f"Kenyatta National Hospital: ({lat:.6f}, {lon:.6f})")

lat, lon = geocode("Tafawa Balewa Square, Lagos")
print(f"Tafawa Balewa Square:       ({lat:.6f}, {lon:.6f})")

lat, lon = geocode("Kampala City Square")
print(f"Kampala City Square:        ({lat:.6f}, {lon:.6f})")

## 2 · Country hint

For ambiguous names (many cities share names) pass an ISO 3166-1 alpha-2
code. This *biases* results — it does not hard-filter them.


In [ ]:
# "Lagos" alone might resolve to Lagos, Portugal or Lagos, Nigeria
lat_ng, lon_ng = geocode("Lagos", country_hint="NG")
lat_no_hint, lon_no_hint = geocode("Lagos")

print(f"Lagos (country_hint='NG'): ({lat_ng:.4f}, {lon_ng:.4f})")
print(f"Lagos (no hint):           ({lat_no_hint:.4f}, {lon_no_hint:.4f})")
print()
# Nigeria is between ~3°–15° E longitude
print("Nigeria result?", 2 < lon_ng < 15)

## 3 · `TravelTimeRequest.from_place()`

Geocode and construct the request in a single call — the most ergonomic
path for interactive use.


In [ ]:
from idd_travel_time import viz
from idd_travel_time.constants import TimeContext, TransportMode
from idd_travel_time.isochrone import compute
from idd_travel_time.models import TransitProfile, TravelTimeRequest

# Walking isochrone from a named hospital
req = TravelTimeRequest.from_place(
    "Mulago National Referral Hospital, Kampala",
    TransportMode.WALKING,
    duration_minutes=15,
)

print(f"Origin resolved to: ({req.origin_lat:.6f}, {req.origin_lon:.6f})")
print(f"Mode: {req.mode.value}, Duration: {req.duration_minutes} min")

In [ ]:
result = compute(req)
viz.render(result)

## 4 · Transit request from place name


In [ ]:
req_transit = TravelTimeRequest.from_place(
    "Lagos Island",
    TransportMode.TRANSIT,
    country_hint="NG",
    duration_minutes=30,
    time_context=TimeContext.RUSH_HOUR,
    transit_profile=TransitProfile.from_preset(
        "informal_minibus",
        time_context=TimeContext.RUSH_HOUR,
        typical_speed_kph=(10.0, 18.0),
    ),
)

print(f"Origin: ({req_transit.origin_lat:.4f}, {req_transit.origin_lon:.4f})")
print(f"Transit profile: {req_transit.transit_profile.mode_name}")

## 5 · `geocode_many()` — batch geocoding

Returns a list of `(lat, lon)` tuples in the same order as the input.
Failed lookups produce `None` when `on_error="warn"` (the default).


In [ ]:
import warnings

from idd_travel_time.locations import geocode_many

cities = [
    "Lagos Island, Nigeria",
    "Nairobi CBD, Kenya",
    "Kampala, Uganda",
    "Abuja, Nigeria",
    "Kigali, Rwanda",
    "this_place_does_not_exist_xyz",  # deliberate failure
]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    coords = geocode_many(cities, on_error="warn")

for city, coord in zip(cities, coords, strict=False):
    if coord:
        print(f"  ✓  {city:<35} ({coord[0]:.4f}, {coord[1]:.4f})")
    else:
        print(f"  ✗  {city:<35} (not found)")

if caught:
    print(f"\n{len(caught)} warning(s) emitted for failed lookups.")

## 6 · The geocoding cache

Results are cached in memory by default — the second call for the same
query is instant. Set `IDD_GEOCODE_CACHE_DIR` to persist to disk across
sessions.


In [ ]:
import time

# First call — hits Nominatim
t0 = time.monotonic()
geocode("Nairobi CBD, Kenya")
t1 = time.monotonic()

# Second call — served from cache
t2 = time.monotonic()
geocode("Nairobi CBD, Kenya")
t3 = time.monotonic()

print(f"First call (network):  {(t1 - t0) * 1000:.0f} ms")
print(f"Second call (cache):   {(t3 - t2) * 1000:.1f} ms")

In [ ]:
# Persist cache to disk — survives kernel restarts and batch job restarts
import os
import tempfile
from pathlib import Path

cache_dir = Path(tempfile.mkdtemp()) / "geocode_cache"
os.environ["IDD_GEOCODE_CACHE_DIR"] = str(cache_dir)

# Reload the module cache so it picks up the env var
import idd_travel_time.locations.geocode as _gc

_gc._cache = None  # reset to force re-init with disk cache

geocode("Kigali, Rwanda")
cache_file = cache_dir / "geocode_cache.json"
print(f"Cache file exists: {cache_file.exists()}")
print(f"Cache contents: {cache_file.read_text()}")

## 7 · Handling geocoding failures gracefully

Use `on_error="raise"` when you need all lookups to succeed (e.g. a
validated input list). Use `on_error="warn"` when partial results are
acceptable (e.g. exploratory batch runs).


In [ ]:
from idd_travel_time.locations import GeocodingError

# on_error="raise" — fails fast on the first bad entry
good_cities = ["Nairobi, Kenya", "Kampala, Uganda"]
coords = geocode_many(good_cities, on_error="raise")
print("All resolved:", coords)

# Catch individual failures
try:
    geocode("xyzzy_not_a_place_12345")
except GeocodingError as exc:
    print(f"\nGeocodingError caught: {exc}")